# 02 - Bias and Fairness Analysis
**NovaCred Credit Application Governance Analysis**

## 1. Imports & Load Data 

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

from scipy import stats
from dateutil import parser as date_parser


df = pd.read_csv("../data/cleaned_credit_applications.csv")
df_raw = df.copy()
df_raw.head()

,app_id,full_name,email,ssn,ip_address,gender,date_of_birth,zip_code,annual_income,credit_history_months,...,processing_timestamp,loan_purpose,notes,gender_original,date_of_birth_original,high_dti_flag,email_valid,completeness_score,completeness_pct,ssn_duplicate_flag
0,app_200,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155,Male,2001-03-09,10036.0,73000.0,23.0,...,2024-01-15T00:00:00Z,NaN,NaN,Male,2001-03-09,False,True,12,100.0,False
1,app_037,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112,Male,1992-03-31,10032.0,78000.0,51.0,...,NaN,NaN,NaN,M,1992-03-31,False,True,12,100.0,False
2,app_215,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250,Male,1989-10-24,10075.0,61000.0,41.0,...,NaN,vacation,NaN,Male,1989-10-24,False,True,12,100.0,False
3,app_024,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67,Male,1983-04-25,10077.0,103000.0,70.0,...,NaN,NaN,NaN,Male,1983-04-25,False,True,12,100.0,False
4,app_184,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105,Male,1999-05-21,10080.0,57000.0,14.0,...,2024-01-15T00:00:00Z,NaN,NaN,M,1999-05-21,False,True,12,100.0,False


---
## 2. PII Field Identification & Classification

Under GDPR Article 4(1), personal data is any information that relates to an identified or identifiable natural person. The analysis below distinguishes between **direct identifiers** (unambiguous linkage to an individual), **quasi-identifiers** (combinatorial re-identification risk), and **sensitive attributes** as defined by Article 9.

> **Reference:** GDPR Art. 4(1); Recital 26 (anonymisation standard); WP29/EDPB Opinion 05/2014 on anonymisation techniques.

In [3]:
#  PII Classification 

PII_CATALOGUE = [
    # ── Direct Identifiers ────────────────────────────────────────────────────
    {
        'column'          : 'full_name',
        'pii_category'    : 'Direct Identifier',
        'gdpr_article'    : 'Art. 4(1)',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Pseudonymise with HMAC-SHA256; remove from analytical layer',
        'retention'       : '5 years post-contract closure (AML obligation)',
    },
    {
        'column'          : 'email',
        'pii_category'    : 'Direct Identifier',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(c) — data minimisation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Pseudonymise for analytics; plaintext only in CRM with access controls',
        'retention'       : 'Duration of consent or contractual relationship',
    },
    {
        'column'          : 'ssn',
        'pii_category'    : 'Direct Identifier — Government ID',
        'gdpr_article'    : 'Art. 87; Recital 75 — high-risk identifier',
        'risk_level'      : 'CRITICAL',
        'treatment'       : 'Tokenise; never expose in analytical datasets; access restricted to compliance team only',
        'retention'       : '5 years post-contract (AML/KYC Directive 2015/849)',
    },
    {
        'column'          : 'ip_address',
        'pii_category'    : 'Online Identifier',
        'gdpr_article'    : 'Art. 4(1); Recital 30; CJEU C-582/14 (Breyer ruling)',
        'risk_level'      : 'MEDIUM',
        'treatment'       : 'Mask last octet (x.x.x.0); retain masked version max 90 days for fraud detection',
        'retention'       : '90 days — legitimate interest (Art. 6(1)(f)); LIA required',
    },
    # ── Quasi-Identifiers ────────────────────────────────────────────────────
    {
        'column'          : 'date_of_birth',
        'pii_category'    : 'Direct / Quasi-Identifier',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(c) — minimisation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Generalise to age band (e.g. 25–34) for modelling; retain full DOB only where legally required',
        'retention'       : 'Governed by contractual/legal necessity',
    },
    {
        'column'          : 'date_of_birth_original',
        'pii_category'    : 'Direct / Quasi-Identifier (raw source field)',
        'gdpr_article'    : 'Art. 4(1), Art. 5(1)(e) — storage limitation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Delete after transformation to date_of_birth; no justification for retaining both fields',
        'retention'       : 'Should not persist beyond initial ingestion pipeline',
    },
    {
        'column'          : 'zip_code',
        'pii_category'    : 'Quasi-Identifier',
        'gdpr_article'    : 'Recital 26 — re-identification risk when combined with other fields',
        'risk_level'      : 'MEDIUM',
        'treatment'       : 'Aggregate to regional level for analytics; validate k-anonymity (k ≥ 5)',
        'retention'       : 'Duration of analytical purpose',
    },
    {
        'column'          : 'gender',
        'pii_category'    : 'Quasi-Identifier / Art. 9 proximity',
        'gdpr_article'    : 'Art. 9(1) — may reveal gender identity; Art. 5(1)(b) — purpose limitation',
        'risk_level'      : 'HIGH',
        'treatment'       : 'Assess necessity for credit model; if retained, document explicit lawful basis and conduct DPIA',
        'retention'       : 'Requires explicit consent (Art. 9(2)(a)) or statutory exception',
    },
    {
        'column'          : 'processing_timestamp',
        'pii_category'    : 'Metadata / Indirect Identifier',
        'gdpr_article'    : 'Art. 4(1) — identifiable when linked to applicant record',
        'risk_level'      : 'LOW',
        'treatment'       : 'Retain for audit trail (Art. 5(2) accountability); truncate to date only in analytical layer',
        'retention'       : '7 years for audit purposes',
    },
]

pii_df = pd.DataFrame(PII_CATALOGUE)

print(f"PII fields identified: {len(pii_df)}")
print(f"\nRisk distribution:")
print(pii_df['risk_level'].value_counts().to_string())
print()
print(pii_df[['column', 'pii_category', 'risk_level', 'gdpr_article']].to_string(index=False))

PII fields identified: 9

Risk distribution:
risk_level
HIGH        5
MEDIUM      2
CRITICAL    1
LOW         1

                column                                 pii_category risk_level                                                              gdpr_article
             full_name                            Direct Identifier       HIGH                                                                 Art. 4(1)
                 email                            Direct Identifier       HIGH                               Art. 4(1), Art. 5(1)(c) — data minimisation
                   ssn            Direct Identifier — Government ID   CRITICAL                                Art. 87; Recital 75 — high-risk identifier
            ip_address                            Online Identifier     MEDIUM                      Art. 4(1); Recital 30; CJEU C-582/14 (Breyer ruling)
         date_of_birth                    Direct / Quasi-Identifier       HIGH                                    Art. 4(1

---
## 3. Privacy Techniques

GDPR Recital 26 and Article 25 (Data Protection by Design) require that personal data be processed in a manner that reduces re-identification risk. This section demonstrates three complementary techniques:

| Technique | How it works | Reversible? | GDPR Status |
|-----------|-------------|-------------|-------------|
| **Pseudonymisation — Salted Hashing** | Replace identifiers with a keyed hash; salt prevents brute-force reversal | Yes (with key) | Still personal data — Art. 4(5) |
| **k-Anonymity — Generalisation & Suppression** | Generalise quasi-identifiers so each record is indistinguishable from ≥ k−1 others | No | Not personal data (if k is sufficient) |
| **Differential Privacy — Laplace Mechanism** | Add calibrated noise to outputs so an individual's presence cannot be inferred | No (mathematical guarantee) | Not personal data |
#### 
> **Warning** Pseudonymisation ≠ Anonymisation. A pseudonymised record can still be re-linked with the mapping key — it remains personal data under GDPR.

### Technique 1: Pseudonymisation via Salted Hashing

HMAC (keyed hash) is the GDPR-recommended approach for pseudonymisation. The pseudonym is deterministic (same input → same token) but irreversible without the secret key. Key must be stored separately (Art. 25 + Rec. 78).

In [10]:
import hashlib, secrets

# Generate a secret salt — in production this lives in a secrets manager / HSM
SALT = secrets.token_hex(16)
print(f"Salt (store separately — never in the dataset): {SALT[:8]}...  [REDACTED]\n")

def salted_hash(value: str, salt: str) -> str:
    """SHA-256 with salt — prevents brute-force reversal"""
    return hashlib.sha256((salt + str(value)).encode('utf-8')).hexdigest()[:16]

df_pseudo = df_raw.copy()

PII_COLS = ['full_name', 'email', 'ssn', 'ip_address']
existing_pii = [c for c in PII_COLS if c in df_pseudo.columns]

for col in existing_pii:
    df_pseudo[f'{col}_pseudonym'] = df_pseudo[col].apply(lambda x: salted_hash(str(x), SALT))
    df_pseudo.drop(columns=[col], inplace=True)

print(f"Columns pseudonymised: {existing_pii}")
print(f"Replaced with       : {[c + '_pseudonym' for c in existing_pii]}\n")

# Show before vs after side by side
pseudonym_cols = [c + '_pseudonym' for c in existing_pii]
before = df_raw[existing_pii].head()
after  = df_pseudo[pseudonym_cols].head()

print("Before pseudonymisation:")
display(before)
print("After pseudonymisation:")
display(after)

Salt (store separately — never in the dataset): 03c0e755...  [REDACTED]

Columns pseudonymised: ['full_name', 'email', 'ssn', 'ip_address']
Replaced with       : ['full_name_pseudonym', 'email_pseudonym', 'ssn_pseudonym', 'ip_address_pseudonym']

Before pseudonymisation:


,full_name,email,ssn,ip_address
0,Jerry Smith,jerry.smith17@hotmail.com,596-64-4340,192.168.48.155
1,Brandon Walker,brandon.walker2@yahoo.com,425-69-4784,10.1.102.112
2,Scott Moore,scott.moore94@mail.com,370-78-5178,10.240.193.250
3,Thomas Lee,thomas.lee6@protonmail.com,194-35-1833,192.168.175.67
4,Brian Rodriguez,brian.rodriguez86@aol.com,480-41-2475,172.29.125.105


After pseudonymisation:


,full_name_pseudonym,email_pseudonym,ssn_pseudonym,ip_address_pseudonym
0,8a02d7f64c67583a,d2349f2be2eac51c,1618e446837a9ef3,5b4c6c8599162f2b
1,bd131dd320d59c05,8680cbd9e3d89c48,28bc82521d423109,3b627f284116b337
2,e2534f86cdd04a70,1f46cbd9b80d7530,b364467f8150b1e4,54bb3d7ca8699415
3,de73fba6e2e6c2cd,a80af1339018415a,426bcbd1549687d3,51ec9294055ccecf
4,a5a419b8cdc65533,501eda9c2e5035fd,876d202307b3f5c6,2d196a9a862c75fa
